# 02 Temporal Patterns

Build hourly, daily, and monthly temporal demand profiles and export them for the Streamlit Temporal Patterns page.

Exports:
- temporal_patterns_hourly.csv
- temporal_patterns_daily.csv
- temporal_patterns_monthly.csv

In [ ]:
import sys
import pandas as pd

sys.path.insert(0, '..')
from utils import load_app_ready, export_df

In [ ]:
df = load_app_ready()
df.shape

In [ ]:
required = ['city_name', 'year', 'start_hour', 'day_of_week', 'day_name', 'month', 'month_name', 'trip_id']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns for temporal analysis: {missing}')

base = df.dropna(subset=['city_name', 'year']).copy()
base['year'] = pd.to_numeric(base['year'], errors='coerce')
base = base.dropna(subset=['year'])
base['year'] = base['year'].astype(int)

hourly = (
    base.dropna(subset=['start_hour', 'day_of_week', 'day_name'])
    .groupby(['city_name', 'year', 'day_of_week', 'day_name', 'start_hour'], as_index=False)
    .agg(trips=('trip_id', 'count'))
    .sort_values(['city_name', 'year', 'day_of_week', 'start_hour'])
)

daily = (
    base.dropna(subset=['day_of_week', 'day_name'])
    .groupby(['city_name', 'year', 'day_of_week', 'day_name'], as_index=False)
    .agg(trips=('trip_id', 'count'))
    .sort_values(['city_name', 'year', 'day_of_week'])
)

monthly = (
    base.dropna(subset=['month', 'month_name'])
    .groupby(['city_name', 'year', 'month', 'month_name'], as_index=False)
    .agg(trips=('trip_id', 'count'))
    .sort_values(['city_name', 'year', 'month'])
)

hourly.head(), daily.head(), monthly.head()

In [ ]:
export_df('temporal_patterns_hourly', hourly)
export_df('temporal_patterns_daily', daily)
export_df('temporal_patterns_monthly', monthly)
print('Exported temporal pattern tables.')